# Analisis
Análisis de variantes. Finalmente, utiliza herramientas como Ensembl VEP (https://grch37.ensembl.org/info/docs/tools/vep/index.html) para determinar la patogenicidad de las variantes encontradas, su significado biológico y más información.

In [1]:
def filter(fname):
    import csv
    from collections import Counter, defaultdict

    vcf_in = f"../results/{fname}.vcf"
    gtf_file = "../data/Homo_sapiens.GRCh37.87.gtf"

    vcf_out = f"../results/{fname}_prioritized_splice30.vcf"
    csv_out = f"../results/{fname}_prioritized_splice30_summary.csv"

    ONLY_PAH = True
    TARGET_GENE = "PAH"
    SPLICE_WINDOW = 30

    KEEP_IMPACTS = {"HIGH", "MODERATE"}
    KEEP_CONSEQUENCES = {
        "splice_donor_variant",
        "splice_acceptor_variant",
        "splice_region_variant",
    }

    def parse_gtf_attributes(attr_text):
        attrs = {}
        for item in attr_text.strip().split(";"):
            item = item.strip()
            if not item:
                continue
            if " " in item:
                key, value = item.split(" ", 1)
                attrs[key] = value.strip('"')
        return attrs

    def load_exon_boundaries_for_gene(gtf_path, gene_symbol):
        """
        Devuelve por cromosoma un conjunto de posiciones de límites exón-intrón.
        Para detectar variantes intrónicas cercanas a splice sites, se usan
        inicio y fin de cada exón como puntos de referencia.
        """
        boundaries = defaultdict(set)

        with open(gtf_path) as f:
            for line in f:
                if line.startswith("#"):
                    continue

                fields = line.rstrip("\n").split("\t")
                if len(fields) < 9:
                    continue

                chrom, source, feature, start, end, score, strand, frame, attrs_text = fields

                if feature != "exon":
                    continue

                attrs = parse_gtf_attributes(attrs_text)
                gene_name = attrs.get("gene_name") or attrs.get("gene_id")

                if gene_name != gene_symbol:
                    continue

                start = int(start)
                end = int(end)

                chrom_variants = {chrom}
                if chrom.startswith("chr"):
                    chrom_variants.add(chrom.replace("chr", "", 1))
                else:
                    chrom_variants.add("chr" + chrom)

                for c in chrom_variants:
                    boundaries[c].add(start)
                    boundaries[c].add(end)

        return boundaries

    def distance_to_nearest_boundary(chrom, pos, boundaries):
        if chrom not in boundaries or not boundaries[chrom]:
            return None

        return min(abs(pos - b) for b in boundaries[chrom])

    def get_info_value(info, key):
        prefix = key + "="
        for item in info.split(";"):
            if item.startswith(prefix):
                return item[len(prefix):]
        return ""

    print("Cargando límites exón-intrón de PAH...")
    boundaries = load_exon_boundaries_for_gene(gtf_file, TARGET_GENE)
    print("Cromosomas con límites cargados:", list(boundaries.keys()))

    csq_fields = []
    idx = {}

    total_variants = 0
    kept_variants = 0

    reason_counter = Counter()
    impact_counter = Counter()
    consequence_counter = Counter()
    clinvar_counter = Counter()

    rows = []

    with open(vcf_in) as fin, open(vcf_out, "w") as fout:
        for line in fin:
            if line.startswith("##INFO=<ID=CSQ"):
                fmt = line.split("Format: ", 1)[1].split('">', 1)[0]
                csq_fields = fmt.split("|")
                idx = {name: i for i, name in enumerate(csq_fields)}
                fout.write(line)

            elif line.startswith("#"):
                fout.write(line)

            else:
                total_variants += 1
                fields = line.rstrip("\n").split("\t")
                chrom, pos, var_id, ref, alt, qual, filt, info = fields[:8]
                pos_int = int(pos)

                csq_value = get_info_value(info, "CSQ")
                if not csq_value:
                    continue

                keep_variant = False
                variant_rows = []

                nearest_splice_distance = distance_to_nearest_boundary(
                    chrom, pos_int, boundaries
                )

                for ann in csq_value.split(","):
                    values = ann.split("|")

                    def get(field):
                        i = idx.get(field)
                        return values[i] if i is not None and i < len(values) else ""

                    symbol = get("SYMBOL")
                    impact = get("IMPACT")
                    consequence = get("Consequence")
                    hgvsc = get("HGVSc")
                    hgvsp = get("HGVSp")
                    intron = get("INTRON")
                    exon = get("EXON")
                    clin_sig = get("ClinVar_CLNSIG") or get("CLIN_SIG")
                    clin_disease = get("ClinVar_CLNDN")
                    clin_review = get("ClinVar_CLNREVSTAT")
                    clin_hgvs = get("ClinVar_CLNHGVS")
                    existing = get("Existing_variation")

                    if ONLY_PAH and symbol != TARGET_GENE:
                        continue

                    consequences = set(consequence.split("&")) if consequence else set()

                    keep_by_impact = impact in KEEP_IMPACTS
                    keep_by_splice_annotation = bool(consequences & KEEP_CONSEQUENCES)

                    # Variante intrónica cerca de límite exón-intrón
                    keep_by_splice30 = (
                            "intron_variant" in consequences
                            and nearest_splice_distance is not None
                            and nearest_splice_distance <= SPLICE_WINDOW
                    )

                    if keep_by_impact or keep_by_splice_annotation or keep_by_splice30:
                        keep_variant = True

                        if keep_by_impact:
                            reason = "impact_HIGH_MODERATE"
                        elif keep_by_splice_annotation:
                            reason = "VEP_splice_annotation"
                        else:
                            reason = "intronic_within_30bp_splice_site"

                        reason_counter[reason] += 1
                        impact_counter[impact] += 1
                        for c in consequences:
                            consequence_counter[c] += 1
                        if clin_sig:
                            clinvar_counter[clin_sig] += 1

                        variant_rows.append({
                            "CHROM": chrom,
                            "POS": pos,
                            "ID": var_id,
                            "REF": ref,
                            "ALT": alt,
                            "QUAL": qual,
                            "FILTER": filt,
                            "SYMBOL": symbol,
                            "IMPACT": impact,
                            "Consequence": consequence,
                            "EXON": exon,
                            "INTRON": intron,
                            "Nearest_splice_boundary_distance_bp": nearest_splice_distance,
                            "HGVSc": hgvsc,
                            "HGVSp": hgvsp,
                            "Existing_variation": existing,
                            "ClinVar_CLNSIG": clin_sig,
                            "ClinVar_CLNDN": clin_disease,
                            "ClinVar_CLNREVSTAT": clin_review,
                            "ClinVar_CLNHGVS": clin_hgvs,
                            "Reason": reason,
                        })

                if keep_variant:
                    kept_variants += 1
                    fout.write(line)
                    rows.extend(variant_rows)

    with open(csv_out, "w", newline="") as f:
        fieldnames = [
            "CHROM", "POS", "ID", "REF", "ALT", "QUAL", "FILTER",
            "SYMBOL", "IMPACT", "Consequence", "EXON", "INTRON",
            "Nearest_splice_boundary_distance_bp",
            "HGVSc", "HGVSp", "Existing_variation",
            "ClinVar_CLNSIG", "ClinVar_CLNDN", "ClinVar_CLNREVSTAT",
            "ClinVar_CLNHGVS", "Reason",
        ]
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

    print("Total variantes:", total_variants)
    print("Variantes conservadas:", kept_variants)
    print("VCF filtrado:", vcf_out)
    print("CSV resumen:", csv_out)

    print("\nRazones de selección:")
    for k, v in reason_counter.most_common():
        print(k, v)

    print("\nImpactos:")
    for k, v in impact_counter.most_common():
        print(k, v)

    print("\nConsecuencias:")
    for k, v in consequence_counter.most_common():
        print(k, v)

    print("\nClinVar:")
    for k, v in clinvar_counter.most_common():
        print(k, v)

In [2]:
filter("vep_clinvar_freebayes_no_snpeff")
filter("vep_clinvar_varscan_no_snpeff")

Cargando límites exón-intrón de PAH...
Cromosomas con límites cargados: ['12', 'chr12']
Total variantes: 405
Variantes conservadas: 5
VCF filtrado: ../results/vep_clinvar_freebayes_no_snpeff_prioritized_splice30.vcf
CSV resumen: ../results/vep_clinvar_freebayes_no_snpeff_prioritized_splice30_summary.csv

Razones de selección:
impact_HIGH_MODERATE 6
intronic_within_30bp_splice_site 6
VEP_splice_annotation 4

Impactos:
HIGH 6
MODIFIER 6
LOW 4

Consecuencias:
intron_variant 10
frameshift_variant 6
non_coding_transcript_variant 5
splice_polypyrimidine_tract_variant 4
splice_region_variant 4

ClinVar:
Benign 6
Cargando límites exón-intrón de PAH...
Cromosomas con límites cargados: ['12', 'chr12']
Total variantes: 2051
Variantes conservadas: 138
VCF filtrado: ../results/vep_clinvar_varscan_no_snpeff_prioritized_splice30.vcf
CSV resumen: ../results/vep_clinvar_varscan_no_snpeff_prioritized_splice30_summary.csv

Razones de selección:
intronic_within_30bp_splice_site 269
impact_HIGH_MODERATE 15

In [3]:
def score(input_fname, output_fname):
    import csv

    input_vcf = f"../results/{input_fname}.vcf"
    output_csv = f"../results/{output_fname}.csv"

    TARGET_GENE = "PAH"

    def parse_csq_header(line):
        fmt = line.split("Format: ", 1)[1].split('">', 1)[0]
        return fmt.split("|")

    def get_info_value(info, key):
        prefix = key + "="
        for item in info.split(";"):
            if item.startswith(prefix):
                return item[len(prefix):]
        return ""

    def score_variant(row):
        score = 0
        reasons = []

        clnsig = row.get("ClinVar_CLNSIG", "")
        impact = row.get("IMPACT", "")
        consequence = row.get("Consequence", "")

        if "Pathogenic" in clnsig and "Likely" not in clnsig:
            score += 5
            reasons.append("ClinVar pathogenic")
        elif "Likely_pathogenic" in clnsig or "Likely pathogenic" in clnsig:
            score += 4
            reasons.append("ClinVar likely pathogenic")
        elif "Benign" in clnsig:
            score -= 3
            reasons.append("ClinVar benign")

        if impact == "HIGH":
            score += 4
            reasons.append("HIGH impact")
        elif impact == "MODERATE":
            score += 3
            reasons.append("MODERATE impact")
        elif impact == "LOW":
            score += 1
            reasons.append("LOW impact")

        if any(term in consequence for term in [
            "stop_gained",
            "frameshift_variant",
            "splice_donor_variant",
            "splice_acceptor_variant",
        ]):
            score += 4
            reasons.append("severe functional consequence")

        if "splice_region_variant" in consequence:
            score += 2
            reasons.append("splice region")

        if "missense_variant" in consequence:
            score += 2
            reasons.append("missense")

        if "intron_variant" in consequence:
            score += 1
            reasons.append("intronic candidate")

        return score, "; ".join(reasons)

    def priority(row):
        score = int(row.get("Final_score") or 0)
        impact = row.get("IMPACT", "")
        consequence = row.get("Consequence", "")
        clnsig = row.get("ClinVar_CLNSIG", "")

        bonus = 0

        if "Pathogenic" in clnsig:
            bonus += 100
        elif "Benign" in clnsig:
            bonus -= 50

        if impact == "HIGH":
            bonus += 50
        elif impact == "MODERATE":
            bonus += 40
        elif impact == "LOW":
            bonus += 10

        if "frameshift_variant" in consequence or "stop_gained" in consequence:
            bonus += 50
        if "splice_donor_variant" in consequence or "splice_acceptor_variant" in consequence:
            bonus += 50
        if "splice_region_variant" in consequence:
            bonus += 30
        if "missense_variant" in consequence:
            bonus += 20

        return score + bonus

    csq_fields = []
    idx = {}
    best = {}

    vcf_lines = 0
    csq_annotations = 0
    pah_annotations = 0

    with open(input_vcf) as f:
        for line in f:
            if line.startswith("##INFO=<ID=CSQ"):
                csq_fields = parse_csq_header(line)
                idx = {name: i for i, name in enumerate(csq_fields)}
                continue

            if line.startswith("#"):
                continue

            vcf_lines += 1

            fields = line.rstrip("\n").split("\t")
            chrom, pos, var_id, ref, alt, qual, filt, info = fields[:8]

            csq_value = get_info_value(info, "CSQ")
            if not csq_value:
                continue

            key = (chrom, pos, ref, alt)

            for ann in csq_value.split(","):
                csq_annotations += 1
                values = ann.split("|")

                def get(field):
                    i = idx.get(field)
                    return values[i] if i is not None and i < len(values) else ""

                symbol = get("SYMBOL")
                if symbol != TARGET_GENE:
                    continue

                pah_annotations += 1

                row = {
                    "CHROM": chrom,
                    "POS": pos,
                    "REF": ref,
                    "ALT": alt,
                    "SYMBOL": symbol,
                    "IMPACT": get("IMPACT"),
                    "Consequence": get("Consequence"),
                    "HGVSc": get("HGVSc"),
                    "HGVSp": get("HGVSp"),
                    "Existing_variation": get("Existing_variation"),
                    "ClinVar_CLNSIG": get("ClinVar_CLNSIG") or get("CLIN_SIG"),
                    "ClinVar_CLNDN": get("ClinVar_CLNDN"),
                    "ClinVar_CLNREVSTAT": get("ClinVar_CLNREVSTAT"),
                    "ClinVar_CLNHGVS": get("ClinVar_CLNHGVS"),
                }

                score, reasons = score_variant(row)
                row["Final_score"] = score
                row["Final_priority_reasons"] = reasons

                if score >= 8:
                    row["Final_priority"] = "High"
                elif score >= 4:
                    row["Final_priority"] = "Medium"
                else:
                    row["Final_priority"] = "Low"

                if key not in best or priority(row) > priority(best[key]):
                    best[key] = row

    rows = sorted(
        best.values(),
        key=lambda r: (r["CHROM"], int(r["POS"]), r["REF"], r["ALT"])
    )

    fieldnames = [
        "CHROM", "POS", "REF", "ALT", "SYMBOL",
        "IMPACT", "Consequence", "HGVSc", "HGVSp",
        "Existing_variation",
        "ClinVar_CLNSIG", "ClinVar_CLNDN",
        "ClinVar_CLNREVSTAT", "ClinVar_CLNHGVS",
        "Final_score", "Final_priority", "Final_priority_reasons",
    ]

    with open(output_csv, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

    print("Input VCF:", input_vcf)
    print("Líneas VCF:", vcf_lines)
    print("Anotaciones CSQ totales:", csq_annotations)
    print("Anotaciones PAH:", pah_annotations)
    print("Variantes únicas PAH:", len(rows))
    print("Guardado:", output_csv)

    for row in sorted(rows, key=lambda r: int(r["Final_score"]), reverse=True):
        print(
            row["Final_score"],
            row["Final_priority"],
            row["CHROM"],
            row["POS"],
            row["REF"],
            row["ALT"],
            row["IMPACT"],
            row["Consequence"],
            row["HGVSc"],
            row["HGVSp"],
            row["ClinVar_CLNSIG"],
        )

In [4]:
score("vep_clinvar_freebayes_no_snpeff_prioritized_splice30",
      "vep_clinvar_freebayes_no_snpeff_prioritized_scored_dedup")

Input VCF: ../results/vep_clinvar_freebayes_no_snpeff_prioritized_splice30.vcf
Líneas VCF: 5
Anotaciones CSQ totales: 33
Anotaciones PAH: 33
Variantes únicas PAH: 5
Guardado: ../results/vep_clinvar_freebayes_no_snpeff_prioritized_scored_dedup.csv
8 High chr12 103237428 CTTTCTC CTTCTC HIGH frameshift_variant ENST00000307000.2:c.1179del ENSP00000303500.2:p.Val394Ter 
8 High chr12 103245500 AGGGCA AGGCA HIGH frameshift_variant ENST00000307000.2:c.861del ENSP00000303500.2:p.Leu288CysfsTer48 
8 High chr12 103246657 AGGCC AGCC HIGH frameshift_variant ENST00000307000.2:c.762del ENSP00000303500.2:p.Phe255SerfsTer81 
4 Medium chr12 103311006 AGGGAGG AGGAGG LOW splice_region_variant&splice_polypyrimidine_tract_variant&intron_variant&non_coding_transcript_variant ENST00000546708.1:n.493-4del  
-2 Low chr12 103271350 G A MODIFIER intron_variant ENST00000307000.2:c.338-22C>T  Benign


In [5]:
score("vep_clinvar_varscan_no_snpeff_prioritized_splice30", "vep_clinvar_varscan_no_snpeff_prioritized_scored_dedup")

Input VCF: ../results/vep_clinvar_varscan_no_snpeff_prioritized_splice30.vcf
Líneas VCF: 138
Anotaciones CSQ totales: 878
Anotaciones PAH: 878
Variantes únicas PAH: 138
Guardado: ../results/vep_clinvar_varscan_no_snpeff_prioritized_scored_dedup.csv
15 High chr12 103237555 G T HIGH stop_gained&splice_region_variant ENST00000307000.2:c.1053C>A ENSP00000303500.2:p.Tyr351Ter Pathogenic
14 High chr12 103271240 AG A HIGH frameshift_variant&splice_region_variant ENST00000307000.2:c.425del ENSP00000303500.2:p.Pro142LeufsTer48 Likely_pathogenic
13 High chr12 103237469 AG A HIGH frameshift_variant ENST00000307000.2:c.1138del ENSP00000303500.2:p.Leu380SerfsTer15 Pathogenic
13 High chr12 103237495 AT A HIGH frameshift_variant ENST00000307000.2:c.1112del ENSP00000303500.2:p.Asn371IlefsTer24 Pathogenic
13 High chr12 103237523 AG A HIGH frameshift_variant ENST00000307000.2:c.1084del ENSP00000303500.2:p.Leu362TrpfsTer33 Pathogenic
13 High chr12 103238154 GC G HIGH frameshift_variant ENST00000307000.2:

In [6]:
import pandas as pd

freebayes_file = "../results/vep_clinvar_freebayes_no_snpeff_prioritized_scored_dedup.csv"
varscan_file = "../results/vep_clinvar_varscan_no_snpeff_prioritized_scored_dedup.csv"

clinvar_main_file = "../results/pah_main_clinvar_variants.csv"
clinvar_top_file = "../results/pah_main_clinvar_variants_top10.csv"
clinvar_overlap_file = "../results/pah_clinvar_overlap_freebayes_varscan.csv"

fb = pd.read_csv(freebayes_file)
vs = pd.read_csv(varscan_file)

fb["Caller"] = "FreeBayes"
vs["Caller"] = "VarScan"

combined = pd.concat([fb, vs], ignore_index=True)

for col in ["ClinVar_CLNSIG", "ClinVar_CLNDN", "ClinVar_CLNREVSTAT", "HGVSc", "HGVSp"]:
    if col not in combined.columns:
        combined[col] = ""
    combined[col] = combined[col].fillna("").astype(str)

combined["Final_score"] = pd.to_numeric(combined["Final_score"], errors="coerce").fillna(0)

combined["variant_key"] = (
        combined["CHROM"].astype(str) + ":" +
        combined["POS"].astype(str) + ":" +
        combined["REF"].astype(str) + ">" +
        combined["ALT"].astype(str)
)


def has_relevant_clinvar(row):
    clnsig = row["ClinVar_CLNSIG"]
    clndn = row["ClinVar_CLNDN"]

    if clnsig == "" or clnsig.lower() == "nan":
        return False

    if "Benign" in clnsig and "Pathogenic" not in clnsig:
        return False

    return (
            "Pathogenic" in clnsig
            or "Likely_pathogenic" in clnsig
            or "Likely pathogenic" in clnsig
            or "Phenylketonuria" in clndn
            or "PAH-related" in clndn
    )


clinvar = combined[combined.apply(has_relevant_clinvar, axis=1)].copy()

priority_order = {
    "High": 3,
    "Medium": 2,
    "Low": 1
}


def clinvar_rank(clnsig):
    clnsig = str(clnsig)

    if "Pathogenic" in clnsig and "Likely" not in clnsig and "Benign" not in clnsig:
        return 5
    if "Pathogenic/Likely_pathogenic" in clnsig:
        return 4
    if "Likely_pathogenic" in clnsig or "Likely pathogenic" in clnsig:
        return 4
    if "Pathogenic" in clnsig:
        return 3
    if "Uncertain" in clnsig:
        return 1
    return 0


def consequence_rank(consequence):
    consequence = str(consequence)

    if "stop_gained" in consequence:
        return 5
    if "splice_acceptor_variant" in consequence or "splice_donor_variant" in consequence:
        return 5
    if "frameshift_variant" in consequence:
        return 4
    if "missense_variant" in consequence:
        return 3
    if "splice_region_variant" in consequence:
        return 2
    return 0


clinvar["priority_rank"] = clinvar["Final_priority"].map(priority_order).fillna(0)
clinvar["clinvar_rank"] = clinvar["ClinVar_CLNSIG"].apply(clinvar_rank)
clinvar["consequence_rank"] = clinvar["Consequence"].apply(consequence_rank)


def merge_callers(series):
    return "+".join(sorted(set(series.dropna())))


def keep_best_variant(group):
    group = group.copy()

    group = group.sort_values(
        by=[
            "clinvar_rank",
            "priority_rank",
            "Final_score",
            "consequence_rank"
        ],
        ascending=[False, False, False, False]
    )

    best = group.iloc[0].copy()
    best["Caller"] = merge_callers(group["Caller"])

    return best


main = (
    clinvar
    .groupby("variant_key", group_keys=False)
    .apply(keep_best_variant)
    .reset_index(drop=True)
)

main = main.sort_values(
    by=[
        "clinvar_rank",
        "priority_rank",
        "Final_score",
        "consequence_rank"
    ],
    ascending=[False, False, False, False]
)

cols = [
    "Caller",
    "CHROM", "POS", "REF", "ALT",
    "SYMBOL",
    "IMPACT",
    "Consequence",
    "HGVSc",
    "HGVSp",
    "Existing_variation",
    "ClinVar_CLNSIG",
    "ClinVar_CLNDN",
    "ClinVar_CLNREVSTAT",
    "ClinVar_CLNHGVS",
    "Final_score",
    "Final_priority",
    "Final_priority_reasons",
    "variant_key"
]

cols = [c for c in cols if c in main.columns]

main = main[cols]

main.to_csv(clinvar_main_file, index=False)

top10 = main.head(10)
top10.to_csv(clinvar_top_file, index=False)

if "variant_key" not in main.columns:
    main["variant_key"] = (
            main["CHROM"].astype(str) + ":" +
            main["POS"].astype(str) + ":" +
            main["REF"].astype(str) + ">" +
            main["ALT"].astype(str)
    )

fb_keys = set(
    fb["CHROM"].astype(str) + ":" +
    fb["POS"].astype(str) + ":" +
    fb["REF"].astype(str) + ">" +
    fb["ALT"].astype(str)
)

vs_keys = set(
    vs["CHROM"].astype(str) + ":" +
    vs["POS"].astype(str) + ":" +
    vs["REF"].astype(str) + ">" +
    vs["ALT"].astype(str)
)

overlap_keys = fb_keys & vs_keys

overlap_clinvar = main[main["variant_key"].isin(overlap_keys)].copy()
overlap_clinvar.to_csv(clinvar_overlap_file, index=False)

print("===== SUMMARY =====")
print("FreeBayes total:", len(fb))
print("VarScan total:", len(vs))
print("Combined total:", len(combined))
print("ClinVar relevant variants before dedup:", len(clinvar))
print("ClinVar relevant variants after dedup:", len(main))
print("ClinVar overlap variants:", len(overlap_clinvar))

print("\nSaved files:")
print(clinvar_main_file)
print(clinvar_top_file)
print(clinvar_overlap_file)

print("\nCaller distribution:")
print(main["Caller"].value_counts())

print("\nPriority distribution:")
print(main["Final_priority"].value_counts(dropna=False))

print("\nTop ClinVar variants:")
print(
    main[
        [
            "Caller", "CHROM", "POS", "REF", "ALT",
            "Consequence", "HGVSc", "HGVSp",
            "ClinVar_CLNSIG", "Final_score", "Final_priority"
        ]
    ]
    .head(15)
    .to_string(index=False)
)

===== SUMMARY =====
FreeBayes total: 5
VarScan total: 138
Combined total: 143
ClinVar relevant variants before dedup: 19
ClinVar relevant variants after dedup: 19
ClinVar overlap variants: 0

Saved files:
../results/pah_main_clinvar_variants.csv
../results/pah_main_clinvar_variants_top10.csv
../results/pah_clinvar_overlap_freebayes_varscan.csv

Caller distribution:
Caller
VarScan    19
Name: count, dtype: int64

Priority distribution:
Final_priority
High      18
Medium     1
Name: count, dtype: int64

Top ClinVar variants:
 Caller CHROM       POS REF ALT                       Consequence                        HGVSc                                HGVSp ClinVar_CLNSIG  Final_score Final_priority
VarScan chr12 103237555   G   T stop_gained&splice_region_variant  ENST00000307000.2:c.1053C>A        ENSP00000303500.2:p.Tyr351Ter     Pathogenic           15           High
VarScan chr12 103260442   C   T           splice_acceptor_variant ENST00000307000.2:c.427-1G>A                           

In [7]:
import pandas as pd

clinvar_file = "../results/pah_main_clinvar_variants.csv"

df = pd.read_csv(clinvar_file)

for col in ["Consequence", "ClinVar_CLNSIG", "HGVSc", "HGVSp", "ClinVar_CLNDN"]:
    df[col] = df[col].fillna("").astype(str)

df["Final_score"] = pd.to_numeric(df["Final_score"], errors="coerce").fillna(0)

cols = [
    "Caller",
    "CHROM", "POS", "REF", "ALT",
    "IMPACT",
    "Consequence",
    "HGVSc",
    "HGVSp",
    "ClinVar_CLNSIG",
    "ClinVar_CLNDN",
    "Final_score",
    "Final_priority"
]

cols = [c for c in cols if c in df.columns]

lof = df[
    df["Consequence"].str.contains(
        "stop_gained|frameshift_variant|splice_acceptor_variant|splice_donor_variant",
        case=False,
        regex=True
    )
].copy()

lof = lof.sort_values(
    by=["Final_score"],
    ascending=False
)

missense = df[
    df["Consequence"].str.contains("missense_variant", case=False, regex=True)
    &
    df["ClinVar_CLNSIG"].str.contains("Pathogenic|Likely_pathogenic|Likely pathogenic", case=False, regex=True)
    ].copy()

missense = missense.sort_values(
    by=["Final_score"],
    ascending=False
)

likely_pathogenic = df[
    df["ClinVar_CLNSIG"].str.contains(
        "Likely_pathogenic|Likely pathogenic|Pathogenic/Likely_pathogenic",
        case=False,
        regex=True
    )
].copy()

likely_pathogenic = likely_pathogenic.sort_values(
    by=["Final_score"],
    ascending=False
)

secondary = df[
    df["Final_priority"].astype(str).str.contains("Medium", case=False, regex=True)
].copy()

secondary = secondary.sort_values(
    by=["Final_score"],
    ascending=False
)

print("\n\n=== TABLA 1. Variantes de pérdida de función ===")
print(lof[cols].to_markdown(index=False))

print("\n\n=== TABLA 2. Variantes missense patogénicas ===")
print(missense[cols].to_markdown(index=False))

print("\n\n=== TABLA 3. Variantes likely pathogenic / pathogenic-likely ===")
print(likely_pathogenic[cols].to_markdown(index=False))

print("\n\n=== TABLA 4. Variantes secundarias de prioridad media ===")
print(secondary[cols].to_markdown(index=False))

lof[cols].to_csv("../results/table1_loss_of_function.csv", index=False)
missense[cols].to_csv("../results/table2_pathogenic_missense.csv", index=False)
likely_pathogenic[cols].to_csv("../results/table3_likely_pathogenic.csv", index=False)
secondary[cols].to_csv("../results/table4_secondary_medium_priority.csv", index=False)

print("\n\nSaved:")
print("../results/table1_loss_of_function.csv")
print("../results/table2_pathogenic_missense.csv")
print("../results/table3_likely_pathogenic.csv")
print("../results/table4_secondary_medium_priority.csv")



=== TABLA 1. Variantes de pérdida de función ===
| Caller   | CHROM   |       POS | REF   | ALT   | IMPACT   | Consequence                              | HGVSc                        | HGVSp                                | ClinVar_CLNSIG    | ClinVar_CLNDN                |   Final_score | Final_priority   |
|:---------|:--------|----------:|:------|:------|:---------|:-----------------------------------------|:-----------------------------|:-------------------------------------|:------------------|:-----------------------------|--------------:|:-----------------|
| VarScan  | chr12   | 103237555 | G     | T     | HIGH     | stop_gained&splice_region_variant        | ENST00000307000.2:c.1053C>A  | ENSP00000303500.2:p.Tyr351Ter        | Pathogenic        | not_provided&Phenylketonuria |            15 | High             |
| VarScan  | chr12   | 103271240 | AG    | A     | HIGH     | frameshift_variant&splice_region_variant | ENST00000307000.2:c.425del   | ENSP00000303500.2:p.Pro142Leuf